# Reuse CCM neighbor searches across paired target shifts
Implement a notebook-local function and compare it with the current native lag
scan on the same inputs, at the newly selected library length **L=300**. The main
check uses three existing records, 100 paired draws and 31 shifts in both directions;
two timing passes reverse the execution order.

Only the lag scan is optimized. Original Tp=0/surrogate calculations, production
scripts and `toolbox/` remain unchanged; no GPU backend is used.


## 1. Identify the work that can be shared
For X -> Y, the source target is X(t + Tp) and the embedding is Y's history.
With common anchors and the same library seed, changing Tp changes target values
but not the embedding geometry or sampled library.

| Stage | Depends on Tp here? | This experiment |
|---|---|---|
| Draw library indices from the seed | No | Leave native RNG unchanged |
| Build KDTree and query distances/local indices | No | Reuse exact native query output |
| Map indices; exclude self and nearby times | No, under fixed anchors | Still execute native code |
| Choose E+1 neighbors; handle zero distance; compute weights | No | Still execute native code |
| Read shifted targets, predict, calculate rho | Yes | Execute for every shift |

Reference: installed PyEDM 2.5.7 `CCM.py`, `Project`, `_ccm_for_libsize`,
and `_nan_safe_pearson`; [public CCM interface](https://sugiharalab.github.io/EDM_Documentation/ccm_/).
The native query has k=4+1+2*10=25 candidates before exclusion, then uses four
neighbors. Query reuse preserves native tie ordering and zero-distance handling.


In [ ]:
import os
for name in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS"]:
    os.environ[name] = "1"

from pathlib import Path
from importlib.metadata import version
from types import SimpleNamespace
import importlib
import json
import platform
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyEDM
from IPython.display import display
from threadpoolctl import threadpool_limits
from toolbox.causal import _ccm_lag_peak

PROCESSES = ["linear", "nonlinear", "independent"]  # One existing history per form.
LIBRARY_SIZE = 300              # Fixed number of embedding states; user-selected future setting.
N_LIBRARIES = 100               # Paired random libraries, without replacement within each draw.
LIBRARY_SEED = 20261309         # Production lag-screen master seed.
E = 3                          # Embedding dimension.
TAU = -1                       # Historical embedding spacing in grid steps.
EXCLUSION_RADIUS = 10          # Temporal exclusion in steps (200 years).
LAG_STEPS = np.arange(-15, 16)  # Target shifts, -300 to +300 years.
DT_KYR = 0.020                 # Grid spacing, kyr.
N_TIMING_PASSES = 2            # Reverse backend order on the second pass.
DIRECTIONS = ["X -> Y", "Y -> X"]

ROOT = Path.cwd()
assert (ROOT / "toolbox/causal.py").is_file(), "Run from the project root."
INPUT_DIR = ROOT / "data/processed/02_check_causal_pipeline"
OUTPUT_DIR = ROOT / "data/processed/02_7_check_ccm_neighbor_reuse"
FIGURE_DIR = ROOT / "figures/02_7_check_ccm_neighbor_reuse"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
assert version("pyEDM") == "2.5.7", "Recheck native internals before changing PyEDM version."
ccm_native = importlib.import_module("pyEDM.CCM")
ORIGINAL_KDTREE = ccm_native.KDTree
library_seeds = np.random.default_rng(LIBRARY_SEED).integers(1, 2**31 - 1, N_LIBRARIES)
scan_settings = dict(E=E, tau=TAU, exclusion_radius=EXCLUSION_RADIUS,
                     library_size=LIBRARY_SIZE, lag_steps=LAG_STEPS,
                     library_seeds=library_seeds)

from toolbox.plotting import add_panel_labels


## 2. Load the same numerical inputs once
Both implementations receive the same in-memory DataFrame, avoiding any CSV
rounding difference between the two paths. L=300 is about 21.9% of the 1371 common
anchors in this experiment, rather than exactly one quarter.


In [ ]:
series = {}
for process in PROCESSES:
    path = INPUT_DIR / f"r0_{process}_original_uniform_series.csv"
    frame = pd.read_csv(path)
    assert len(frame) == 1401 and np.isfinite(frame[["time", "age", "x", "y"]]).all().all()
    np.testing.assert_allclose(np.diff(frame["time"]), DT_KYR, rtol=0, atol=1e-10)
    np.testing.assert_allclose(np.diff(frame["age"]), -DT_KYR, rtol=0, atol=1e-10)
    np.testing.assert_allclose(frame["age"].iloc[[0, -1]], [29, 1])
    series[process] = frame
display(pd.DataFrame([{"process": p, "points": len(f), "first_x": f.x.iloc[0],
                       "first_y": f.y.iloc[0]} for p, f in series.items()]))

def common_scan_inputs(frame, *, E, tau, library_size, lag_steps):
    '''Require finite regular inputs and hold prediction/library anchors fixed.'''
    values = frame[["time", "age", "x", "y"]].to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("This prototype requires finite input series.")
    spacing = np.diff(values[:, 0])
    if not len(spacing) or np.any(spacing <= 0) or not np.allclose(spacing, spacing[0]):
        raise ValueError("Time must increase regularly.")
    if not np.allclose(np.diff(values[:, 1]), -spacing):
        raise ValueError("BP age must decrease as time advances.")
    if np.any(np.ptp(values[:, 2:], axis=0) == 0):
        raise ValueError("Both series must be nonconstant.")
    lags = np.asarray(lag_steps)
    if (E < 1 or tau >= 0 or lags.ndim != 1 or len(lags) < 3
            or not np.issubdtype(lags.dtype, np.integer)
            or np.any(np.diff(lags) <= 0) or not lags[0] < 0 < lags[-1] or 0 not in lags):
        raise ValueError("Require historical embedding and an ordered negative/zero/positive lag grid.")
    history = (E - 1) * abs(tau)
    anchors = np.arange(max(history, -int(lags[0])), len(frame) - int(lags[-1]))
    if not E + 2 <= library_size <= len(anchors):
        raise ValueError("Library does not fit the common anchor set.")
    valid_lib = np.zeros(len(frame), dtype=bool)
    valid_lib[anchors] = True
    return frame[["time", "x", "y"]], anchors, valid_lib


## 3. Retain the current native scan as the reference
The reference loops over shifts first, reusing one native object for each shift and
calling `Project(sample=1, seed=...)` for each library. Its full rho array is kept;
no mean over libraries replaces the individual paired curves.


In [ ]:
def scan_ccm_native(frame, *, E, tau, exclusion_radius, library_size, lag_steps, library_seeds):
    data, anchors, valid_lib = common_scan_inputs(
        frame, E=E, tau=tau, library_size=library_size, lag_steps=lag_steps)
    rho = np.full((len(library_seeds), len(lag_steps), 2), np.nan)
    for lag_index, lag in enumerate(lag_steps):
        model = None
        for draw, seed in enumerate(library_seeds):
            if model is None:
                model = pyEDM.CCM(
                    dataFrame=data, columns="y", target="x", E=E, tau=tau, Tp=int(lag),
                    exclusionRadius=exclusion_radius, libSizes=[library_size],
                    sample=1, seed=int(seed), validLib=valid_lib, parallel=False,
                    includeData=False, showPlot=False, returnObject=True)
                np.testing.assert_array_equal(model.idx_col, anchors)
                np.testing.assert_array_equal(model.idx_tgt, anchors)
            else:
                model.Project(sample=1, seed=int(seed), parallel=False, includeData=False)
            assert int(model.libMeans["LibSize"].iloc[0]) == library_size
            rho[draw, lag_index] = model.libMeans[["y:x", "x:y"]].iloc[0].to_numpy()
    return {"rho": rho, "anchors": anchors, "library_seeds": np.array(library_seeds),
            "lag_steps": np.array(lag_steps), "directions": np.array(DIRECTIONS)}


## 4. Reuse only exact native queries, one library at a time
The new function loops over libraries first and retains one native object per
shift. A local cache matches the complete library/query array bytes, shapes, dtypes
and k; misses use the original SciPy KDTree and hits return its unchanged output.

The cache is cleared for each draw and at function exit. The temporary module
patch is restored in `finally`, including on errors; this notebook prototype must
run sequentially, never concurrently in threads. The narrow wrapper accepts only
the constructor/query arguments actually used by PyEDM 2.5.7.


In [ ]:
def scan_ccm_reusing_neighbors(frame, *, E, tau, exclusion_radius, library_size,
                               lag_steps, library_seeds):
    '''Reuse native KDTree queries across shifts; keep all estimator arithmetic native.'''
    data, anchors, valid_lib = common_scan_inputs(
        frame, E=E, tau=tau, library_size=library_size, lag_steps=lag_steps)
    native_tree = ccm_native.KDTree
    query_cache = {}
    stats = {"query_requests": 0, "native_searches": 0, "cache_hits": 0,
             "peak_cache_entries": 0, "peak_query_array_bytes": 0, "peak_key_bytes": 0}

    def array_key(array):
        contiguous = np.ascontiguousarray(array)
        return contiguous.shape, contiguous.dtype.str, contiguous.tobytes()

    def cached_tree(library):
        library_key = array_key(library)

        def query(points, k):
            stats["query_requests"] = stats["query_requests"] + 1
            query_key = array_key(points)
            key = (library_key, query_key, int(k))
            if key in query_cache:
                stats["cache_hits"] = stats["cache_hits"] + 1
                return query_cache[key]
            # Delay both construction and query until an actual cache miss.
            result = native_tree(library).query(points, k=k)
            for array in result:
                array.flags.writeable = False
            query_cache[key] = result
            stats["native_searches"] = stats["native_searches"] + 1
            stats["peak_cache_entries"] = max(stats["peak_cache_entries"], len(query_cache))
            query_bytes = sum(a.nbytes for pair in query_cache.values() for a in pair)
            key_bytes = sum(len(key[0][2]) + len(key[1][2]) for key in query_cache)
            stats["peak_query_array_bytes"] = max(stats["peak_query_array_bytes"], query_bytes)
            stats["peak_key_bytes"] = max(stats["peak_key_bytes"], key_bytes)
            return result

        return SimpleNamespace(query=query)

    rho = np.full((len(library_seeds), len(lag_steps), 2), np.nan)
    models = [None] * len(lag_steps)
    try:
        ccm_native.KDTree = cached_tree
        for draw, seed in enumerate(library_seeds):
            query_cache.clear()
            for lag_index, lag in enumerate(lag_steps):
                model = models[lag_index]
                if model is None:
                    model = pyEDM.CCM(
                        dataFrame=data, columns="y", target="x", E=E, tau=tau, Tp=int(lag),
                        exclusionRadius=exclusion_radius, libSizes=[library_size],
                        sample=1, seed=int(seed), validLib=valid_lib, parallel=False,
                        includeData=False, showPlot=False, returnObject=True)
                    np.testing.assert_array_equal(model.idx_col, anchors)
                    np.testing.assert_array_equal(model.idx_tgt, anchors)
                    models[lag_index] = model
                else:
                    model.Project(sample=1, seed=int(seed), parallel=False, includeData=False)
                assert int(model.libMeans["LibSize"].iloc[0]) == library_size
                rho[draw, lag_index] = model.libMeans[["y:x", "x:y"]].iloc[0].to_numpy()
            assert len(query_cache) <= 2, "Expected fixed geometry for the two directions."
    finally:
        ccm_native.KDTree = native_tree
        query_cache.clear()
    return {"rho": rho, "anchors": anchors, "library_seeds": np.array(library_seeds),
            "lag_steps": np.array(lag_steps), "directions": np.array(DIRECTIONS),
            "cache_stats": stats}


## 5. Check intermediate arrays, repeated states and cleanup
A small instrumented check captures the first two raw neighbor searches and every
prediction/target vector for two libraries and three shifts. It also checks a
repeated-state series with exact zero distances, one saved uncertain chronology
pair, and restoration after an injected query failure; none are timing measurements.


In [ ]:
def peak_table(result):
    rows = []
    for draw, curves in enumerate(result["rho"]):
        for direction, name in enumerate(DIRECTIONS):
            lag, rho, status = _ccm_lag_peak(curves[:, direction], result["lag_steps"])
            rows.append({"draw": draw, "direction": name, "peak_step": lag,
                         "peak_rho": rho, "status": status})
    return pd.DataFrame(rows)

def compare_scans(reference, candidate):
    for key in ["rho", "anchors", "library_seeds", "lag_steps", "directions"]:
        np.testing.assert_array_equal(reference[key], candidate[key], err_msg=key)
    pd.testing.assert_frame_equal(peak_table(reference), peak_table(candidate), check_exact=True)

def capture_native_vectors(scan, frame, settings):
    '''Capture diagnostics outside timing; all numerical operations remain native.'''
    native_tree, native_pearson = ccm_native.KDTree, ccm_native._nan_safe_pearson
    searches, predictions, actuals = [], [], []

    def observed_tree(library):
        tree = native_tree(library)
        def query(points, k):
            result = tree.query(points, k=k)
            if len(searches) < 2:
                searches.append((library.copy(), points.copy(), result[0].copy(), result[1].copy()))
            return result
        return SimpleNamespace(query=query)

    def observed_pearson(predicted, actual):
        predictions.append(predicted.copy())
        actuals.append(actual.copy())
        return native_pearson(predicted, actual)

    try:
        ccm_native.KDTree = observed_tree
        ccm_native._nan_safe_pearson = observed_pearson
        result = scan(frame, **settings)
    finally:
        ccm_native.KDTree = native_tree
        ccm_native._nan_safe_pearson = native_pearson
    n_draws, n_lags = len(settings["library_seeds"]), len(settings["lag_steps"])
    ordered = []
    for vectors in [predictions, actuals]:
        if scan is scan_ccm_native:
            array = np.array(vectors).reshape(n_lags, n_draws, 2, -1).transpose(1, 0, 2, 3)
        else:
            array = np.array(vectors).reshape(n_draws, n_lags, 2, -1)
        ordered.append(array)
    return result, searches, ordered

small_settings = dict(scan_settings, library_seeds=library_seeds[:2],
                      lag_steps=LAG_STEPS[[0, len(LAG_STEPS) // 2, -1]])
repeated = series["linear"].copy()
repeated["x"] = np.resize(np.array([0., 1., 0., -1.]), len(repeated))
repeated["y"] = np.resize(np.array([1., 0., -1., 0.]), len(repeated))
audit_rows, audit_arrays = [], {}
with threadpool_limits(limits=1):
    for name, frame in [("nonlinear", series["nonlinear"]), ("repeated_states", repeated)]:
        reference, ref_search, ref_vectors = capture_native_vectors(scan_ccm_native, frame, small_settings)
        candidate, new_search, new_vectors = capture_native_vectors(scan_ccm_reusing_neighbors, frame, small_settings)
        compare_scans(reference, candidate)
        for direction, (expected, actual) in enumerate(zip(ref_search, new_search)):
            for field, a, b in zip(["library", "queries", "distances", "indices"], expected, actual):
                np.testing.assert_array_equal(a, b)
                audit_arrays[f"{name}_direction{direction}_{field}"] = a
        for field, expected, actual in zip(["predictions", "targets"], ref_vectors, new_vectors):
            np.testing.assert_array_equal(expected, actual)
            audit_arrays[f"{name}_native_{field}"] = expected
            audit_arrays[f"{name}_reused_{field}"] = actual
        zero_distances = sum(np.count_nonzero(search[2] == 0) for search in ref_search)
        if name == "repeated_states":
            assert zero_distances > 0
        audit_rows.append({"check": name, "all_arrays_exact": True,
                           "prediction_values": ref_vectors[0].size,
                           "zero_distances_in_first_searches": zero_distances})

    uncertain_path = ROOT / "data/processed/02_3_check_uncertainty_escalation/r0_nonlinear_level4_C-J_m0_series.csv"
    uncertain = pd.read_csv(uncertain_path)
    compare_scans(scan_ccm_native(uncertain, **small_settings),
                  scan_ccm_reusing_neighbors(uncertain, **small_settings))
    audit_rows.append({"check": "nonlinear_level4_C-J_m0", "all_arrays_exact": True})

    # Raise inside the patched query path, then verify restoration before cleanup.
    def failing_tree(library):
        def query(points, k):
            raise RuntimeError("Injected query failure")
        return SimpleNamespace(query=query)

    try:
        ccm_native.KDTree = failing_tree
        try:
            scan_ccm_reusing_neighbors(series["linear"], **small_settings)
        except RuntimeError as error:
            assert str(error) == "Injected query failure"
        else:
            raise AssertionError("Injected exception was swallowed.")
        assert ccm_native.KDTree is failing_tree
    finally:
        ccm_native.KDTree = ORIGINAL_KDTREE
    recovery = scan_ccm_native(series["linear"], **small_settings)
    assert np.isfinite(recovery["rho"]).all()
    audit_rows.append({"check": "exception_restore_and_native_recovery", "all_arrays_exact": True})

assert ccm_native.KDTree is ORIGINAL_KDTREE
np.savez_compressed(OUTPUT_DIR / "small_audit_arrays.npz", **audit_arrays)
pd.DataFrame(audit_rows).to_csv(OUTPUT_DIR / "small_checks.csv", index=False)
display(pd.DataFrame(audit_rows))


## 6. Compare every rho and repeat timings in reverse order
The full comparison requires exact numerical equality, including NaN locations,
seeds, anchors and every peak classification. Warm-up is provided by the preceding
small checks; timing includes preparation, cache keys and cleanup, but not saving
files or plotting, and is limited to a sequential CPU lag scan.


In [ ]:
functions = {"native": scan_ccm_native, "reuse": scan_ccm_reusing_neighbors}
results, timing_rows, comparison_rows, cache_rows = {}, [], [], []
with threadpool_limits(limits=1):
    for pass_index in range(N_TIMING_PASSES):
        order = ["native", "reuse"] if pass_index % 2 == 0 else ["reuse", "native"]
        for process in PROCESSES:
            pair = {}
            for backend in order:
                start = time.perf_counter()
                result = functions[backend](series[process], **scan_settings)
                elapsed = time.perf_counter() - start
                assert ccm_native.KDTree is ORIGINAL_KDTREE
                assert np.isfinite(result["rho"]).all()
                results[(pass_index, process, backend)] = result
                pair[backend] = result
                timing_rows.append({"pass": pass_index + 1, "process": process,
                                    "backend": backend, "seconds": elapsed,
                                    "order": " -> ".join(order)})
                # Preserve arrays before comparison so a discrepancy remains inspectable.
                np.savez_compressed(OUTPUT_DIR / f"{process}_{backend}_pass{pass_index + 1}.npz",
                                    **{k: v for k, v in result.items() if k != "cache_stats"})
                if backend == "reuse":
                    stats = result["cache_stats"]
                    assert stats["query_requests"] == 2 * N_LIBRARIES * len(LAG_STEPS)
                    assert stats["native_searches"] == 2 * N_LIBRARIES
                    assert stats["cache_hits"] == 2 * N_LIBRARIES * (len(LAG_STEPS) - 1)
                    assert stats["peak_cache_entries"] == 2
                    cache_rows.append({"pass": pass_index + 1, "process": process, **stats})
                print(f"Pass {pass_index + 1} | {process:11s} | {backend:6s}: {elapsed:.2f} s", flush=True)
            compare_scans(pair["native"], pair["reuse"])
            if pass_index > 0:
                compare_scans(results[(0, process, "native")], pair["native"])
                compare_scans(results[(0, process, "reuse")], pair["reuse"])
            difference = pair["reuse"]["rho"] - pair["native"]["rho"]
            comparison_rows.append({"pass": pass_index + 1, "process": process,
                                    "rho_values": difference.size, "rho_exact": True,
                                    "maximum_absolute_error": float(np.max(np.abs(difference))),
                                    "peak_rows_exact": len(peak_table(pair["native"]))})

timings = pd.DataFrame(timing_rows)
comparisons = pd.DataFrame(comparison_rows)
cache_stats = pd.DataFrame(cache_rows)
timings.to_csv(OUTPUT_DIR / "timings.csv", index=False)
comparisons.to_csv(OUTPUT_DIR / "comparisons.csv", index=False)
cache_stats.to_csv(OUTPUT_DIR / "cache_stats.csv", index=False)
for process in PROCESSES:
    peak_table(results[(0, process, "reuse")]).to_csv(OUTPUT_DIR / f"{process}_peaks.csv", index=False)
settings = {
    "library_size": LIBRARY_SIZE, "n_libraries": N_LIBRARIES, "library_seed": LIBRARY_SEED,
    "E": E, "tau": TAU, "exclusion_radius": EXCLUSION_RADIUS,
    "lag_steps": LAG_STEPS.tolist(), "dt_kyr": DT_KYR, "timing_passes": N_TIMING_PASSES,
    "inputs": [str((INPUT_DIR / f"r0_{p}_original_uniform_series.csv").relative_to(ROOT)) for p in PROCESSES],
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {p: version(p) for p in ["numpy", "scipy", "pandas", "pyEDM"]},
    "scope": "Lag scan only; native query reuse; not full CCM or production throughput.",
}
(OUTPUT_DIR / "settings.json").write_text(json.dumps(settings, indent=2), encoding="utf-8")
display(comparisons)
display(cache_stats)


## 7. Inspect numerical agreement and the measured speedup
The first figure overlays median lag curves and displays the largest absolute
rho difference over all draws at each shift. The second shows both timing passes;
cache payload estimates exclude the 31 retained native models and Python overhead.


In [ ]:
lag_years = LAG_STEPS * DT_KYR * 1000
fig, axes = plt.subplots(len(PROCESSES), 2, figsize=(11, 9), sharex=True, layout="constrained")
direction_colors = ["#4682B4", "#E69F00"]
for row, process in enumerate(PROCESSES):
    reference = results[(0, process, "native")]["rho"]
    candidate = results[(0, process, "reuse")]["rho"]
    for direction, (name, color) in enumerate(zip(DIRECTIONS, direction_colors)):
        axes[row, 0].plot(lag_years, np.median(reference[:, :, direction], axis=0),
                          color=color, label=f"{name}: native")
        axes[row, 0].plot(lag_years, np.median(candidate[:, :, direction], axis=0),
                          color=color, linestyle="--", marker=".", markersize=3,
                          label=f"{name}: reused")
        difference = np.max(np.abs(candidate[:, :, direction] - reference[:, :, direction]), axis=0)
        axes[row, 1].plot(lag_years, difference, color=color, clip_on=False, label=name)
    axes[row, 0].set_title(f"{process.capitalize()} | median rho")
    axes[row, 1].set_title(f"{process.capitalize()} | maximum absolute difference")
    maximum_error = float(np.max(np.abs(candidate - reference)))
    axes[row, 1].set_ylim(0, maximum_error * 1.1 if maximum_error > 0 else 1e-15)
    if maximum_error == 0:
        axes[row, 1].set_yticks([0])
    axes[row, 0].set_ylabel("Cross-map rho")
    axes[row, 1].set_ylabel("Absolute rho difference")
    for ax in axes[row]:
        ax.axvline(0, color="0.6", linewidth=0.8, linestyle=":")
        ax.grid(alpha=0.15)
        if row == len(PROCESSES) - 1:
            ax.set_xlabel("Target shift Tp (years)")
axes[0, 0].legend(fontsize=8, ncol=2)
axes[0, 1].legend(fontsize=8)
fig.suptitle("Native CCM versus exact neighbor-query reuse, L=300")
add_panel_labels(axes)
fig.savefig(FIGURE_DIR / "01_numerical_agreement.png", dpi=180)
fig.savefig(FIGURE_DIR / "01_numerical_agreement.pdf")
plt.show()

timing_pairs = timings.pivot(index=["process", "pass"], columns="backend", values="seconds")
timing_pairs["speedup"] = timing_pairs["native"] / timing_pairs["reuse"]
display(timing_pairs.round(3))
fig, axes = plt.subplots(1, len(PROCESSES), figsize=(11, 3.6), sharey=True, layout="constrained")
for ax, process in zip(axes, PROCESSES):
    group = timing_pairs.loc[process]
    ax.bar([0, 1], [group.native.median(), group.reuse.median()],
           color=["#777777", "#009E73"], width=0.6, alpha=0.8)
    for pass_number, values in group.iterrows():
        offset = (pass_number - (N_TIMING_PASSES + 1) / 2) * 0.12
        ax.scatter(np.array([0, 1]) + offset, [values.native, values.reuse], color="black", s=18, zorder=3)
    ax.set_xticks([0, 1], ["Native", "Reuse"])
    ax.set_title(f"{process.capitalize()} | {group.speedup.median():.2f}x")
    ax.grid(axis="y", alpha=0.2)
axes[0].set_ylabel("Lag-scan wall time (seconds)")
fig.suptitle("Two passes with reversed order; bars show median times")
add_panel_labels(axes)
fig.savefig(FIGURE_DIR / "02_lag_scan_timings.png", dpi=180)
fig.savefig(FIGURE_DIR / "02_lag_scan_timings.pdf")
plt.show()
payload_mib = (cache_stats.peak_query_array_bytes + cache_stats.peak_key_bytes) / 1024**2
print(f"Peak cached query arrays plus key bytes: {payload_mib.max():.3f} MiB.")


## 8. Interpret the result before migration
Exact full-array agreement supports reusing these native queries for the tested
inputs and pinned version. It does not validate concurrent threads, other PyEDM
versions, GPU arithmetic, a full surrogate workflow, or 32-worker throughput.

L=300 is the user's chosen setting for future work. Completed L=1119 results and
production defaults have not been rewritten; a later production change must use
the new library length consistently for original scores, nulls and lag screening,
and begin a scientifically distinct run. The new function remains in this notebook.

**Observed default run (2026-11-07).** All 37,200 pairwise rho comparisons across
three records and two passes were exactly equal (maximum absolute difference 0),
with matching dtypes, seeds, anchors and all 1,200 peak rows. The intermediate
prediction/target arrays, duplicate-state case, uncertain-input check and exception
restoration check also passed. No numerical tolerance was needed.

| Record | Native median time (s) | Reuse median time (s) | Median paired speedup |
|---|---:|---:|---:|
| Linear | 21.21 | 5.58 | 3.81x |
| Nonlinear | 21.82 | 5.46 | 4.00x |
| Independent | 23.00 | 5.58 | 4.12x |

Each scan made 200 real neighbor searches and reused 6,000 query results, while
still calculating all 6,200 rho values. The cache held at most two entries: query
arrays plus key bytes occupied 1.122 MiB at peak, excluding native model objects
and other overhead. Remaining native filtering, weights, predictions and Python
calls explain why the complete lag scan is not 31 times faster. The full notebook
executed in about 169 seconds; full CCM and 32-worker throughput were not measured.
